![DB Academy](./Includes/images/db-academy.png)

# Demo - Adding Metadata to the Knowledge Store

## Overview

In this demo, you will improve a Genie Space by adding table and column descriptions directly in Unity Catalog. Descriptions give Genie the context it needs to understand your data, like what each table contains, what each column means, and what values are valid. 

You will use `ALTER TABLE` SQL statements to add descriptions to all five bakehouse tables at once, then return to Genie to observe how those descriptions affect the quality of its responses. You will also add column synonyms in the Knowledge Store to map business language to column names. 

Finally, you will enable prompt matching features to help Genie map user language to actual data values.

## Learning Objectives

By the end of this demo, you will be able to:

1. **Add table and column descriptions** to Unity Catalog tables using `ALTER TABLE` SQL statements.
2. **Verify descriptions** in Unity Catalog using `DESCRIBE TABLE EXTENDED` and Catalog Explorer.
3. **Add column synonyms** in the Knowledge Store to map business terms to column names.
4. **Evaluate the impact of descriptions** on Genie's SQL generation by re-asking questions from a prior demo.
5. **Distinguish what descriptions solve** from what requires general instructions or example SQL.
6. **Enable prompt matching** (format assistance and entity matching) on key categorical columns in the Knowledge Store.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;">Complete <strong>the previous demonstrations</strong> before proceeding. 

This demo builds on the tables and benchmarks we added to Genie.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select a Serverless SQL Warehouse</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless SQL Warehouse (2X-Small is sufficient)**
  - Select **Compute** > **More** > **SQL Warehouse** > Select a SQL Warehouse you have access to.

**NOTE:** This notebook was **developed and tested using Serverless SQL Warehouse**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

1. Run the cell below to set your default catalog and schema.
    - This assumes you have already run the setup from a prior demo and your tables exist in **labuser_YOUR_USER_NAME.genie_course_bakehouse**.

In [0]:
%run ./Includes/Classroom-Setup-creating-table-column-descriptions

## B. Genie Space Authoring Checklist

General 6-step approach whenever you create a new Genie space. Here we will focus on **Step 3**. 

<br></br>
<!--
  Focus variant of the Genie Space Authoring Checklist.
  Each step card shows its full-color inline styles AND a commented "ACTIVE" version for the dimmed cards.
  To bring a dimmed step back to color: swap the dimmed <div> opening with the line marked "ACTIVE", and restore the inner number/title colors.
-->

<div style="max-width: 1280px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<!-- 6-STEP FLOW -->
<div style="display: grid; grid-template-columns: 1fr auto 1fr auto 1fr auto 1fr auto 1fr auto 1fr; gap: 0; align-items: stretch;">

  <!-- STEP 1 (DIMMED) -->
  <!-- ACTIVE: <div style="background: #F9F7F4; border: 3px solid #2574B5; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;"> -->
  <div style="background: #EEEDE9; border: 3px solid #C2C2C2; border-radius: 10px; padding: 16px 12px; box-shadow: none; text-align: center; min-width: 0; opacity: 0.7;">
    <div style="font-size: 26pt; font-weight: 800; color: #90A5B1; line-height: 1; margin-bottom: 8px;">1</div>
    <div style="font-size: 14pt; font-weight: 600; color: #5A6F77; line-height: 1.35;">Start with purpose<br>and questions</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #C2C2C2; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 2 (DIMMED) -->
  <!-- ACTIVE: <div style="background: #F9F7F4; border: 3px solid #02A36F; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;"> -->
  <div style="background: #EEEDE9; border: 3px solid #C2C2C2; border-radius: 10px; padding: 16px 12px; box-shadow: none; text-align: center; min-width: 0; opacity: 0.7;">
    <div style="font-size: 26pt; font-weight: 800; color: #90A5B1; line-height: 1; margin-bottom: 8px;">2</div>
    <div style="font-size: 14pt; font-weight: 600; color: #5A6F77; line-height: 1.35;">Select a minimal<br>data slice</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #C2C2C2; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 3 (active) -->
  <div style="background: #F9F7F4; border: 3px solid #F8A805; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;">
    <div style="font-size: 26pt; font-weight: 800; color: #F8A805; line-height: 1; margin-bottom: 8px;">3</div>
    <div style="font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35;">Add metadata<br>and synonyms</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #C2C2C2; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 4 (DIMMED) -->
  <!-- ACTIVE: <div style="background: #F9F7F4; border: 3px solid #FF5F46; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;"> -->
  <div style="background: #EEEDE9; border: 3px solid #C2C2C2; border-radius: 10px; padding: 16px 12px; box-shadow: none; text-align: center; min-width: 0; opacity: 0.7;">
    <div style="font-size: 26pt; font-weight: 800; color: #90A5B1; line-height: 1; margin-bottom: 8px;">4</div>
    <div style="font-size: 14pt; font-weight: 600; color: #5A6F77; line-height: 1.35;">Encode business<br>logic in SQL</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #C2C2C2; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 5 (DIMMED) -->
  <!-- ACTIVE: <div style="background: #F9F7F4; border: 3px solid #98182A; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;"> -->
  <div style="background: #EEEDE9; border: 3px solid #C2C2C2; border-radius: 10px; padding: 16px 12px; box-shadow: none; text-align: center; min-width: 0; opacity: 0.7;">
    <div style="font-size: 26pt; font-weight: 800; color: #90A5B1; line-height: 1; margin-bottom: 8px;">5</div>
    <div style="font-size: 14pt; font-weight: 600; color: #5A6F77; line-height: 1.35;">Add light global<br>text instructions</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #C2C2C2; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 6 (DIMMED) -->
  <!-- ACTIVE: <div style="background: #F9F7F4; border: 3px solid #1C3037; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;"> -->
  <div style="background: #EEEDE9; border: 3px solid #C2C2C2; border-radius: 10px; padding: 16px 12px; box-shadow: none; text-align: center; min-width: 0; opacity: 0.7;">
    <div style="font-size: 26pt; font-weight: 800; color: #90A5B1; line-height: 1; margin-bottom: 8px;">6</div>
    <div style="font-size: 14pt; font-weight: 600; color: #5A6F77; line-height: 1.35;">Test, benchmark,<br>and iterate with users</div>
  </div>

</div>

</div>


## C. Add Table and Column Descriptions

Rather than manually adding descriptions one by one in the Catalog Explorer UI, you will use `ALTER TABLE` SQL statements to add all table and column descriptions and primary/foreign key constraints at once.

These are stored directly in **Unity Catalog** and Genie reads them automatically. 

**No additional configuration is required.**

### C1. Add Descriptions and Constraints with `ALTER TABLE`

The SQL below does three things for each of the five bakehouse tables:

| What | Why |
|------|-----|
| **Table descriptions** (`COMMENT ON TABLE`) | Tells Genie what each table represents and when to use it |
| **Column descriptions** (`ALTER COLUMN ... COMMENT`) | Clarifies what each column means, lists valid values, and disambiguates shared column names across tables |
| **Primary / foreign key constraints** (`ADD CONSTRAINT ... NOT ENFORCED`) | Informational metadata that tells Genie how tables join together. They are not enforced at the data level, but Genie reads them to auto-detect join relationships |

1. Run the cell below to add all descriptions and constraints at once instead of adding them manually (to save time). 

    Feel free to explore the metadata being added as the code is executing.

**NOTES:** 

- The code has been prepared for you to avoid manual entry. 
- In production scenarios it's important to add the correct metadata on your tables and columns to give Genie the correct information in the Knowledge store.
- **NOTE:** You can also use **Genie Code** to help create metadata for your tables and columns. You should always review the AI generated metadata for accuracy.
[AWS](https://docs.databricks.com/aws/en/genie-code/) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/genie-code/use-genie-code) |
[GCP](https://docs.databricks.com/gcp/en/genie-code/)


In [0]:
%sql
-- -----------------------------------------------------------
-- sales_customers_gold
-- -----------------------------------------------------------
COMMENT ON TABLE sales_customers_gold
  IS 'Customer profiles for the bakehouse franchise business. Each row represents one unique customer. There are no duplicate or null customerIDs. To count total customers, use COUNT(*) on this table. Use customerID to join with sales_transactions_gold.';

ALTER TABLE sales_customers_gold
  ALTER COLUMN customerID      COMMENT 'Unique identifier for each customer. Primary key — every row has a non-null, unique value. Use COUNT(*) to count total customers. Join key to sales_transactions_gold.customerID.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN first_name      COMMENT 'Customer first name.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN last_name       COMMENT 'Customer last name.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN email_address   COMMENT 'Customer email address.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN phone_number    COMMENT 'Customer phone number.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN address         COMMENT 'Customer street address.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN city            COMMENT 'City where the customer lives. This is the customer city, not the franchise city.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN state           COMMENT 'State or province where the customer lives.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN country         COMMENT 'Country where the customer lives. This is the customer country, not the franchise country.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN continent       COMMENT 'Continent where the customer lives.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN postal_zip_code COMMENT 'Postal or zip code for the customer address.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN gender          COMMENT 'Customer gender. Valid values: female, male.';

ALTER TABLE sales_customers_gold ALTER COLUMN customerID SET NOT NULL;
ALTER TABLE sales_customers_gold ADD CONSTRAINT pk_customers PRIMARY KEY (customerID) NOT ENFORCED;

-- -----------------------------------------------------------
-- sales_suppliers_gold  (must come before franchises so PK exists for FK reference)
-- -----------------------------------------------------------
COMMENT ON TABLE sales_suppliers_gold
  IS 'Ingredient suppliers for bakehouse franchises. Each row represents a supplier that provides a specific ingredient. Use supplierID to join with sales_franchises_gold.';

ALTER TABLE sales_suppliers_gold
  ALTER COLUMN supplierID   COMMENT 'Unique identifier for each supplier. Join key to sales_franchises_gold.supplierID.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN name         COMMENT 'Name of the supplier company.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN ingredient   COMMENT 'The specific ingredient this supplier provides (e.g., almonds, vanilla, cinnamon, honey).';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN continent    COMMENT 'Continent where the supplier is located.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN city         COMMENT 'City where the supplier is located.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN district     COMMENT 'District or neighborhood where the supplier is located.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN size         COMMENT 'Size category of the supplier. Valid values: S (Small), M (Medium), L (Large), XL (Extra Large), XXL (Extra Extra Large).';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN longitude    COMMENT 'Geographic longitude coordinate of the supplier.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN latitude     COMMENT 'Geographic latitude coordinate of the supplier.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN approved     COMMENT 'Approval status of the supplier. Valid values: Y (Yes). All suppliers in this dataset are approved.';

ALTER TABLE sales_suppliers_gold ALTER COLUMN supplierID SET NOT NULL;
ALTER TABLE sales_suppliers_gold ADD CONSTRAINT pk_suppliers PRIMARY KEY (supplierID) NOT ENFORCED;

-- -----------------------------------------------------------
-- sales_franchises_gold  (FK references sales_suppliers_gold PK defined above)
-- -----------------------------------------------------------
COMMENT ON TABLE sales_franchises_gold
  IS 'Bakery franchise locations worldwide. Each row represents a unique franchise with its location, size category, and assigned supplier. Use franchiseID to join with sales_transactions_gold and media_customer_reviews_gold (use the store column to join). Some franchises have the same name but different locations — always include city or franchiseID to uniquely identify a location. IMPORTANT: Many of the franchises do not have a supplierID that matches any record in sales_suppliers_gold when combining the tables. When querying franchise-supplier relationships, always use a LEFT JOIN to include all franchises, not an INNER JOIN.';

ALTER TABLE sales_franchises_gold
  ALTER COLUMN franchiseID COMMENT 'Unique identifier for each franchise location. Join key to sales_transactions_gold.franchiseID and media_customer_reviews_gold.ref. Multiple franchises can share the same name in different locations. Always include franchiseID and city in GROUP BY when aggregating by franchise to avoid merging distinct locations.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN name        COMMENT 'Name of the franchise location. Note: some franchise names are shared across different cities, so always include city and franchiseID to uniquely identify a location.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN city        COMMENT 'City where the franchise is located. This is the franchise city, not the customer city.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN district    COMMENT 'District or neighborhood within the city where the franchise is located.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN zipcode     COMMENT 'Postal or zip code for the franchise location.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN country     COMMENT 'Country where the franchise is located. This is the franchise country, not the customer country. Use this when asked about franchise location or revenue by country. Valid values: US, Canada, Japan, Australia, Netherlands, France, Germany, Italy, Sweden.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN size        COMMENT 'Size category of the franchise. Valid values: S (Small), M (Medium), L (Large), XL (Extra Large), XXL (Extra Extra Large).';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN longitude   COMMENT 'Geographic longitude coordinate of the franchise.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN latitude    COMMENT 'Geographic latitude coordinate of the franchise.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN supplierID  COMMENT 'Foreign key to sales_suppliers_gold.supplierID. IMPORTANT: Only 27 of 48 franchises have a matching supplier. The remaining 21 have supplierIDs that do not exist in sales_suppliers_gold. Always use LEFT JOIN when joining to sales_suppliers_gold to avoid silently dropping unmatched franchises.';

ALTER TABLE sales_franchises_gold ALTER COLUMN franchiseID SET NOT NULL;
ALTER TABLE sales_franchises_gold ADD CONSTRAINT pk_franchises PRIMARY KEY (franchiseID) NOT ENFORCED;
ALTER TABLE sales_franchises_gold ADD CONSTRAINT fk_franchises_supplier FOREIGN KEY (supplierID) REFERENCES sales_suppliers_gold(supplierID) NOT ENFORCED;

-- -----------------------------------------------------------
-- sales_transactions_gold
-- -----------------------------------------------------------
COMMENT ON TABLE sales_transactions_gold
  IS 'Sales transactions for bakehouse franchise products. Each row represents a single purchase. Use customerID and franchiseID to join with customer and franchise tables.';

ALTER TABLE sales_transactions_gold
  ALTER COLUMN transactionID COMMENT 'Unique identifier for each transaction.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN customerID    COMMENT 'Foreign key to sales_customers_gold.customerID. Identifies the customer who made the purchase.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN franchiseID   COMMENT 'Foreign key to sales_franchises_gold.franchiseID. Identifies the franchise where the purchase was made.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN dateTime      COMMENT 'Timestamp of when the transaction occurred.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN product       COMMENT 'Name of the bakehouse product purchased. Valid values: Austin Almond Biscotti, Golden Gate Ginger, Orchard Oasis, Outback Oatmeal, Pearly Pies, Tokyo Tidbits.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN quantity      COMMENT 'Number of units purchased in this transaction.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN unitPrice     COMMENT 'Price per unit of the product.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN totalPrice    COMMENT 'Total price for this transaction (unitPrice * quantity). Use SUM(totalPrice) to calculate revenue or total spend. When ranking customers or franchises by performance, use SUM(totalPrice) as the metric.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN paymentMethod COMMENT 'Payment method used. Valid values: amex, mastercard, visa.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN cardNumber    COMMENT 'Card number used for the transaction.';

ALTER TABLE sales_transactions_gold ALTER COLUMN transactionID SET NOT NULL;
ALTER TABLE sales_transactions_gold ADD CONSTRAINT pk_transactions PRIMARY KEY (transactionID) NOT ENFORCED;
ALTER TABLE sales_transactions_gold ADD CONSTRAINT fk_transactions_customer FOREIGN KEY (customerID) REFERENCES sales_customers_gold(customerID) NOT ENFORCED;
ALTER TABLE sales_transactions_gold ADD CONSTRAINT fk_transactions_franchise FOREIGN KEY (franchiseID) REFERENCES sales_franchises_gold(franchiseID) NOT ENFORCED;

-- -----------------------------------------------------------
-- media_customer_reviews_gold
-- -----------------------------------------------------------
COMMENT ON TABLE media_customer_reviews_gold
  IS 'Customer reviews for bakehouse franchise locations. Each row contains free-text feedback and a sentiment classification. Use the ref column to join with sales_franchises_gold.franchiseID to obtain franchise information. The flag column contains the sentiment classification.';

ALTER TABLE media_customer_reviews_gold
  ALTER COLUMN new_id       COMMENT 'Unique identifier for each review.';
ALTER TABLE media_customer_reviews_gold
  ALTER COLUMN ref          COMMENT 'Foreign key to sales_franchises_gold.franchiseID. Identifies the franchise (store/location) being reviewed. Join this column to sales_franchises_gold.franchiseID to get franchise names and locations. Always include ref, franchise name and city in SELECT and GROUP BY when counting or aggregating reviews per franchise, since franchise names are not unique and we want exact information about the reviews by franchise.';
ALTER TABLE media_customer_reviews_gold
  ALTER COLUMN review       COMMENT 'Free-text customer review. Contains unstructured feedback about the franchise experience.';
ALTER TABLE media_customer_reviews_gold
  ALTER COLUMN review_date  COMMENT 'Date when the review was submitted.';
ALTER TABLE media_customer_reviews_gold
  ALTER COLUMN flag         COMMENT 'Sentiment classification of the review. Valid values: positive, negative, mixed. "Bad" reviews mean negative. "Neutral" reviews mean mixed. "Good or great" means positive. Use this column to filter reviews by sentiment rather than analyzing the raw review text.';

ALTER TABLE media_customer_reviews_gold ALTER COLUMN new_id SET NOT NULL;
ALTER TABLE media_customer_reviews_gold ADD CONSTRAINT pk_reviews PRIMARY KEY (new_id) NOT ENFORCED;
ALTER TABLE media_customer_reviews_gold ADD CONSTRAINT fk_reviews_franchise FOREIGN KEY (ref) REFERENCES sales_franchises_gold(franchiseID) NOT ENFORCED;

2. Run the cell below to verify that the:
   -  Descriptions were added to the table.
   -  The following columns have been set as primary and foreign keys.
      - **pk_transactions**
      - **fk_transactions_franchise**
      - **fk_transactions_customer**

   - **NOTE:** You can use **Catalog Explorer** or run `DESCRIBE TABLE EXTENDED <table_name>` to view each of the table details if you'd like. We will simply look at one.

In [0]:
%sql
DESCRIBE TABLE EXTENDED sales_transactions_gold;


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Metadata is extremely important
  </strong>
  <div style="color:#333;">

Genie sometimes gets it right without metadata, but the answers can be inconsistent across runs. 

**Metadata makes it more reliable and consistent.**

Our training data introduces these concepts, but putting into production relies on the curator to improve the knowledge store for better results.

  </div>
</div>

### C2. View Table and Column Descriptions in the Knowledge Store

The **Knowledge Store** is the centralized configuration surface within your Genie where all curation settings live. 

Table and column descriptions that are added in Unity Catalog are automatically pulled into the Knowledge Store for Genie.

1. Open your **Genie Space** in a new tab. (YOUR_NAME - Bakehouse Sales) that you created previously.

2. Select **Configure** > **Data** tab.

3. Select any table and confirm that (**You might have to refresh your Genie page if you already had Genie open**):
   - The **table description** appears at the top
   - Each **column** has a description, with valid values listed where applicable

**NOTES:** 
   - The Genie Space automatically reads these descriptions from **Unity Catalog** with no additional configuration required. 
   - You can also **override or extend these descriptions** directly in the Knowledge Store for this specific Genie space without modifying the underlying Unity Catalog metadata.

## D. Add Column Display Names and Synonyms in the Knowledge Store

In addition to descriptions, the Knowledge Store supports **synonyms** and **display names**.

- **Synonyms** - Alternate names that map business language to column names. This will help Genie map the data to the different ways users would ask the prompt. 

- **Display names** - Uses a friendly display name for this column. **This won't rename the column in Unity Catalog.**

**NOTE:** These will be configured in the knowledge store UI.  If using a **Metric View** as the source, they can be configured there. 

1. In your **Genie Space**, select **Configure** > **Data** > **sales_transactions_gold**.

   a. Select the **pencil icon** next to the **totalPrice** column.

   b. **Display Name** field - `Total Price`

   c. **Synonyms** field, add: `revenue, sales, total sales, total revenue, income`

   d. Click **Save**.

**NOTE:** This allows Genie to map terms like "revenue" or "sales" to the **totalPrice** column.

2. Add synonyms to the **flag** column.

   a. Select **Data** > **media_customer_reviews_gold**.

   b. Select the **pencil icon** next to the **flag** column.

   c. **Display Name** field - `Review Sentiment`

   d. **Synonyms** field, add:
      `sentiment, summary sentiment, feedback sentiment, review sentiment, opinion, review`

   e. Click **Save**.

**NOTE:** This helps Genie map terms like "negative rating" or "feedback type" to the **flag** column.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Note
  </strong>
  <div style="color:#333;">

For training purposes, we will stop here with just two synonym and display name examples.

Feel free to continue adding synonyms and display names to other columns as needed.

  </div>
</div>


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Metric Views as a source for Genie are recommended
  </strong>
  <div style="color:#333;">

Instead of referencing tables directly in Genie, you can also use Metric Views.

#### Recall, Metric Views centrally define business metrics and semantics across Databricks, ensuring consistent, governed definitions across all tools.

- Metric views let you define display names, formats, and synonyms for dimensions and measures in a YAML file stored in Unity Catalog.

- This gives you a code-managed, reusable semantic layer that Genie and AI/BI dashboards can share.

- In Genie, synonyms are usually configured per space in the knowledge store UI, not through SQL on the underlying tables. Metric views are a good choice when you want synonyms managed in code.

**NOTE:** Metric views are outside the scope of this Genie course. View the documentation below or check out the [Building Semantic Models in Databricks with UC Metric Views](https://www.databricks.com/training/catalog/building-semantic-models-in-databricks-with-uc-metric-views-4962) Databricks training.

- Unity Catalog Metric Views:
[AWS](https://docs.databricks.com/aws/en/metric-views/) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/metric-views/) |
[GCP](https://docs.databricks.com/gcp/en/metric-views/)


  </div>
</div>

## E. Enable Prompt Matching

Beyond descriptions, display names and synonyms, the Knowledge Store offers **prompt matching**.

**Prompt matching** is a set of features that help Genie bridge the gap between how users phrase questions and how data is actually stored.

Prompt matching has two parts:

<br></br>

<div style="max-width: 950px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<div style="display: flex; gap: 20px; justify-content: center;">

<!-- Format Assistance -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden;">
  <div style="background: #4299E0; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 1.6em; font-weight: bold;">Format Assistance</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 1.3em; color: #555; line-height: 1.6; margin-bottom: 14px;">
      Samples a few <strong>representative values</strong> from a column so Genie understands data types and formatting patterns (dates, codes, numeric shapes, etc.).<br/><br/>
      Helps Genie generate literals in the right format and avoid obviously wrong patterns in filters and comparisons.
    </div>
    <div style="background: rgba(66,153,224,0.10); border-left: 4px solid #4299E0; padding: 10px 12px; border-radius: 6px; font-size: 1em;">
      <strong>Use broadly</strong> — enabled by default on most columns. Low cost, high value.
    </div>
  </div>
</div>

<!-- Entity Matching -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden;">
  <div style="background: #00A972; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 1.3em; font-weight: bold;">Entity Matching</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 1.3em; color: #555; line-height: 1.6; margin-bottom: 14px;">
      Builds a <strong>curated list of distinct values</strong> for key categorical columns (e.g., product names, status codes, regions).<br/><br/>
      Lets Genie map user language and typos to real values — so <em>"Florida"</em> correctly becomes <code>state = 'FL'</code> instead of <code>state ILIKE '%Florida%'</code>.
    </div>
    <div style="background: rgba(0,169,114,0.10); border-left: 4px solid #00A972; padding: 10px 12px; border-radius: 6px; font-size: 1em;">
      <strong>Use selectively</strong> — reserve for high-signal categorical columns that appear in user questions.
    </div>
  </div>
</div>

</div>

</div>

Prompt Matching Review - [AWS](https://docs.databricks.com/aws/en/genie/knowledge-store#-prompt-matching-overview) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/genie/knowledge-store#-prompt-matching-overview) |
[GCP](https://docs.databricks.com/gcp/en/genie/knowledge-store#-prompt-matching-overview) 

### E1. View Format Assistance 

Format assistance is **enabled by default** on most columns. Let's confirm it is active.

1. In your Genie Space, select **Configure** > **Data** > **sales_transactions_gold**.

2. Click the **pencil icon** next to the **product** column.

3. Click **Advanced**.

4. Confirm that **Format assistance** is toggled **on**.
   - This means Genie has already sampled representative values from this column (e.g., `Austin Almond Biscotti`, `Golden Gate Ginger`, etc.).
   - These values help Genie understand the formatting pattern and generate correct filter literals.

5. Leave this open.

### E2. Enable Entity Matching on Key Columns

Entity matching goes further than format assistance.

It builds a **complete list of distinct values** for a column so Genie can map user language (including misspellings and synonyms) to exact data values.

**Best practice:** Enable entity matching on a small set of **high-signal categorical columns** that users frequently reference in questions.

For the bakehouse dataset, good candidates would be:

| Table | Column | Why |
|-------|--------|-----|
| `sales_transactions_gold` | `product` | Users ask about specific products by name |
| `media_customer_reviews_gold` | `flag` | Users filter reviews by sentiment |
| `sales_transactions_gold` | `paymentMethod` | Users may ask about payment types |
| `sales_franchises_gold` | `size` | Users ask about franchise size categories |

1. In your Genie Space, select **Configure** > **Data** > **sales_transactions_gold**.

2. Click the **pencil icon** next to the **product** column.

3. Click **Advanced**.

4. Confirm **Format assistance** is on (required for entity matching).

5. Confirm **Entity matching** to **on**.

6. Click **Save**.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Note
  </strong>
  <div style="color:#333;">

For training purposes, we confirmed **entity matching and format assistant** was enabled on a single column. Feel free to enable/disable it on additional categorical columns like `paymentMethod` or `size`. By default it most likely is turned on where necessary.

**Limitations:**
- Entity matching supports only **string columns**
- Up to **120 columns** with up to **1,024 distinct values** per column (max 127 characters per value)
- Cannot be applied to tables with **row filters or column masks**
- **Format assistance must be enabled** before entity matching can be turned on

  </div>
</div>

### E3. Refresh Prompt Matching Data

When new values are added to a column or existing values change format, you should refresh the stored prompt matching data.

In our demonstration it's not necessary.

- View Refresh or remove prompt matching data for more information:
[AWS](https://docs.databricks.com/aws/en/genie/knowledge-store#refresh-or-remove-prompt-matching-data) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/genie/knowledge-store#refresh-or-remove-prompt-matching-data) |
[GCP](https://docs.databricks.com/gcp/en/genie/knowledge-store#refresh-or-remove-prompt-matching-data)

## F. Run Benchmarks

In **an earlier demonstration**, you saved benchmark questions to establish a **baseline** for your Genie Space before any metadata was added. 

Now that you have added table descriptions, column descriptions, synonyms, and prompt matching, let's run those same benchmarks to measure the impact.

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Why Run Benchmarks Now?
  </strong>
  <div style="color:#333;">

Asking questions manually is useful for exploring behavior, but it is **subjective** and **not repeatable**. 

Benchmarks give you a **structured, repeatable way** to measure whether your changes actually improved Genie's accuracy and consistency.

Each time you make a change to your Genie Space: descriptions, instructions, example SQL, you should re-run benchmarks to confirm:
- Did the change **fix** the questions it was intended to fix?
- Did the change **break** anything that was already working?

This is the **test, benchmark, and iterate** step from the authoring checklist.

  </div>
</div>

1. In your Genie Space, select the **Benchmark > Questions** tab in the top navigation.

2. Review the benchmark questions you added during in an earlier demo. 

      You should see up to six questions:

   | # | Benchmark Question | Has Ground Truth SQL? |
   |---|--------------------|-----------------------|
   | 1 | How many customers do we have? | Yes |
   | 2 | What are the top 5 franchise locations by total revenue? | Yes |
   | 3 | Who are our best customers? | Yes |
   | 4 | How many franchises does each supplier serve? | Yes |
   | 5 | What are total sales by region? | Yes |
   | 6 | Count of bad reviews by locations? | Yes |

3. Select **Run all benchmarks** to execute all the benchmark questions.

   - Recall, benchmarks run each question as a **new, independent conversation**
      - **Genie does not carry any context from previous questions or threads**.
      - This simulates how a real user would interact with the space for the first time.

4. As each benchmark completes, review the output. For each question, check:

   - **Assessment** - Did Genie produce a correct response?

   - **Failure analysis** - Is the SQL correct and business-aligned?

   - **Result match** (for questions with ground truth) - Does Genie's output match the expected result? Why or why not? 
      - What else can we add to the knowledge store or metadata to improve our results?

5. **After adding metadata, a few additional benchmarks should pass.**


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Key Takeaways
  </strong>
  <div style="color:#333;">


Metadata and synonyms are the **foundation** of a well-curated Genie Space. 

They are the easiest change to make and often have the biggest immediate impact. 

But as the benchmark results show, some questions require more than metadata:

- **Business terms** like "best customer" or "region" need **general instructions or a SQL query** to define what they mean.

- **Complex query patterns** like `LEFT JOIN` + `IS NULL` need **example SQL** to teach Genie the correct approach.

- **Derived columns** like mapping countries to regions need **SQL expressions** to encode the logic.

You will address these gaps in the next demo. Each time you add a new layer of context, you will re-run these benchmarks to measure progress.

  </div>
</div>



## G. What Metadata Solves and It Doesn't

<br></br>
<div class="two-pane">

<style>

.two-sides {
    display: flex;
    justify-content: center;
    align-items: stretch;
    gap: 40px;
    flex-wrap: wrap;
}

.box {
    width: 480px;
    min-height: 320px;
    background: #F9F7F4;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    display: flex;
    flex-direction: column;
    justify-content: flex-start;
    gap: 14px;
    padding: 22px;
    text-align: left;
    position: relative;
    box-sizing: border-box;
}

.box::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 8px;
}

.box.left::before { background: #00A972; }
.box.right::before { background: #FF5F46; }

.box-title {
    font-size: 20pt;
    font-weight: 700;
    color: #0b2026;
    text-align: center;
    margin-top: 6px;
}

.box-text {
    font-size: 14pt;
    color: #0b2026;
    line-height: 1.55;
}

.box-text ul {
    margin: 0;
    padding-left: 18px;
}

.box-text li {
    margin-bottom: 12px;
}

.box-text li:last-child {
    margin-bottom: 0;
}

</style>

<div class="two-sides">

<div class="box left">
<div class="box-title">✓ What They Solve</div>
<div class="box-text">
<ul>
<li>Clarify what each column contains</li>
<li>List valid values for categorical columns</li>
<li>Clarify shared column names (e.g., customer city vs. franchise city)</li>
<li>Explain join relationships between tables</li>
</ul>
</div>
</div>

<div class="box right">
<div class="box-title">✗ What They Don't Solve</div>
<div class="box-text">
<ul>
<li>Define business terms like "best customer"</li>
<li>Teach complex query patterns like <code>LEFT JOIN</code> + <code>IS NULL</code></li>
<li>Specify default behaviors (e.g., always round to 2 decimals)</li>
<li>Provide reusable SQL templates for common questions</li>
</ul>
</div>
</div>

</div>

</div>

<br></br>

<div style="max-width: 1020px; margin: 0 auto; padding: 18px 24px; background: #F8F9FC; border: 3px solid #1B5162; border-radius: 8px; font-size: 14pt; color: #0b2026; line-height: 1.6;">
  <ul style="margin: 0; padding-left: 20px;">
    <li style="margin-bottom: 12px;">Descriptions and synonyms are the <strong>first and highest-impact lever</strong> for improving a Genie Space</li>
    <li style="margin-bottom: 12px;">Together with <strong>prompt matching</strong> features in the Knowledge Store, they form the metadata foundation of your space</li>
    <li style="margin-bottom: 12px;">For business logic, query patterns, and behavioral rules, you'll need <strong>general instructions</strong> and <strong>example SQL</strong></li>
  </ul>
</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

#### What Descriptions Solve

- **Column clarity** — Genie reads descriptions to understand what a column actually contains, not just its name. A column called `flag` means nothing without a description explaining it holds sentiment values.
- **Valid values** — Listing valid categorical values (e.g., `positive`, `negative`, `mixed`) prevents Genie from guessing or hallucinating filter conditions.
- **Disambiguation** — When two tables both have a `city` column, descriptions clarify which is the customer's city and which is the franchise's city. Without this, Genie may join or filter on the wrong one.
- **Join relationships** — Describing which column is a foreign key and what it joins to helps Genie construct correct multi-table queries without being told explicitly each time.

#### What Descriptions Don't Solve

- **Business definitions** — "Best customer" could mean highest spend, most visits, or longest tenure. Descriptions can't encode that decision. It requires a general instruction or example SQL.
- **Complex query patterns** — `LEFT JOIN` + `IS NULL` to find non-matching rows is a pattern, not a metadata fact. Genie needs an example SQL query to learn it.
- **Default behaviors** — If every revenue query should round to 2 decimals or filter to the current fiscal year, that rule belongs in general instructions, not a column description.
- **Reusable templates** — Recurring questions like "top 10 franchises by revenue last month" are best handled with saved example SQL, not descriptions.

#### The Bigger Picture

**Descriptions are the foundation.**

They're the easiest win and should always come first. But they have a ceiling. 

Once the metadata is clean, the next layer is behavioral: 
- general instructions tell Genie *how* to behave, 
- and example SQL shows it *how* to write specific query patterns. 

</details>


## H. Conclusion

In this demo, you:

1. Used `ALTER TABLE` SQL statements to add table and column descriptions to all five bakehouse tables in Unity Catalog.
2. Verified descriptions using `DESCRIBE TABLE EXTENDED` and Catalog Explorer.
3. Added column synonyms in the Knowledge Store to map business terms like "revenue" and "rating" to the correct columns.
4. Re-asked Genie the same questions from the first demo and observed how descriptions improved SQL accuracy for column values, join logic, and disambiguation.
5. Identified the limits of descriptions. They can clarify data but cannot define business terms or teach complex query patterns.
6. Enabled **prompt matching** (format assistance and entity matching) on key categorical columns to help Genie map user language to actual data values.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>